# IPP Practical File 2

Swanand Wirkar 20251051 BDA

## Problem Statement

Problem 3: Compression Ratio Comparison (RLE vs Huffman)
Load a grayscale image with large smooth regions. Compress it using Run-Length Encoding and Huffman Coding. Calculate bit-length before and after compression and percentage reduction. Explain the results.

In [1]:
import cv2
import numpy as np
import heapq
from collections import Counter
from itertools import count

image = cv2.imread("document.png", cv2.IMREAD_GRAYSCALE)
if image is None:
    raise FileNotFoundError("Place a grayscale document or smooth-region image named document.png in the notebook folder.")
pixels = image.ravel().tolist(); original_bits = len(pixels) * 8

def rle_encode(data):
    out=[]
    if not data: return out
    value=data[0]; run=1
    for p in data[1:]:
        if p == value and run < 255: run += 1
        else: out.append((value,run)); value=p; run=1
    out.append((value,run)); return out

runs = rle_encode(pixels)
rle_bits = len(runs) * 16  # 8 bits for pixel value + 8 bits for run length

def huffman_lengths(data):
    freq=Counter(data)
    if len(freq)==1: return {next(iter(freq)):1}
    uid=count(); heap=[(n,next(uid),sym) for sym,n in freq.items()]; heapq.heapify(heap)
    while len(heap)>1:
        f1,_,a=heapq.heappop(heap); f2,_,b=heapq.heappop(heap)
        heapq.heappush(heap,(f1+f2,next(uid),(a,b)))
    lengths={}
    def visit(node, depth):
        if isinstance(node, tuple): visit(node[0],depth+1); visit(node[1],depth+1)
        else: lengths[node]=depth
    visit(heap[0][2],0); return lengths

freq=Counter(pixels); lengths=huffman_lengths(pixels)
huffman_data_bits=sum(freq[s]*n for s,n in lengths.items())
huffman_overhead=256*16  # illustrative fixed codebook overhead estimate
huffman_bits=huffman_data_bits+huffman_overhead

def reduction(bits): return (original_bits-bits)/original_bits*100
print("Image dimensions:", image.shape)
print("Original:", original_bits, "bits")
print(f"RLE: {len(runs)} runs; {rle_bits} bits; reduction {reduction(rle_bits):.2f}%")
print(f"Huffman: data {huffman_data_bits} bits + estimated codebook {huffman_overhead} bits = {huffman_bits} bits; reduction {reduction(huffman_bits):.2f}%")
print("RLE benefits from long consecutive runs of identical pixels and may expand data when runs are short. Huffman coding assigns shorter codes to frequent values, so its performance depends on pixel-value frequencies. The codebook overhead is an estimate; actual formats store metadata differently.")

Image dimensions: (508, 393)
Original: 1597152 bits
RLE: 44925 runs; 718800 bits; reduction 54.99%
Huffman: data 451577 bits + estimated codebook 4096 bits = 455673 bits; reduction 71.47%
RLE benefits from long consecutive runs of identical pixels and may expand data when runs are short. Huffman coding assigns shorter codes to frequent values, so its performance depends on pixel-value frequencies. The codebook overhead is an estimate; actual formats store metadata differently.
